<a href="https://colab.research.google.com/github/vanashri-18/CSA6101-Digital-Forensics-and-Cybercrime-Investigation/blob/main/File_Activity_Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Aim**

To develop a Python-based file activity recorder that monitors a selected directory for a specified period, detects created, modified, and deleted files, records each event with its timestamp, category, and file path, and generates an activity summary after monitoring is completed.

**Algorithm**

Select the directory to be monitored.

Define the monitoring duration and checking interval.

Record the initial state of files in the directory.

Continuously scan the directory during the monitoring period.

Compare the current state with the previous state.

Detect newly created files.

Detect modified files.

Detect deleted files.

Record every event with timestamp, category, and file path.

Display the recorded events and generate an activity summary.

In [1]:
# ==============================================
# FILE ACTIVITY RECORDER
# ==============================================

import os
import time
import pandas as pd
from datetime import datetime

# ----------------------------------------------
# 1. Configuration
# ----------------------------------------------

MONITOR_DIR = "/content/monitor_folder"
MONITOR_TIME = 30       # seconds
CHECK_INTERVAL = 2      # seconds

os.makedirs(MONITOR_DIR, exist_ok=True)

# ----------------------------------------------
# 2. Function to Get File State
# ----------------------------------------------

def get_file_state():

    state = {}

    for root, dirs, files in os.walk(MONITOR_DIR):

        for file in files:

            path = os.path.join(root, file)

            try:
                state[path] = os.path.getmtime(path)
            except OSError:
                pass

    return state


# ----------------------------------------------
# 3. Initial State
# ----------------------------------------------

previous = get_file_state()
events = []

print("=" * 70)
print("             FILE ACTIVITY RECORDER")
print("=" * 70)

print("\nDirectory :", MONITOR_DIR)
print("Duration  :", MONITOR_TIME, "seconds")
print("\nMonitoring started...")

# ----------------------------------------------
# 4. Monitor Directory
# ----------------------------------------------

start = time.time()

while time.time() - start < MONITOR_TIME:

    time.sleep(CHECK_INTERVAL)

    current = get_file_state()

    old_files = set(previous)
    new_files = set(current)

    # Created files
    for path in new_files - old_files:

        events.append([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "CREATED",
            path
        ])

        print("[CREATED] ", path)

    # Deleted files
    for path in old_files - new_files:

        events.append([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "DELETED",
            path
        ])

        print("[DELETED] ", path)

    # Modified files
    for path in old_files & new_files:

        if current[path] != previous[path]:

            events.append([
                datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "MODIFIED",
                path
            ])

            print("[MODIFIED]", path)

    previous = current

# ----------------------------------------------
# 5. Create Activity Report
# ----------------------------------------------

print("\n" + "=" * 70)
print("                 ACTIVITY SUMMARY")
print("=" * 70)

if events:

    report = pd.DataFrame(
        events,
        columns=["Timestamp", "Category", "File_Path"]
    )

    print("\nTotal Events:", len(report))

    print("\nEvent Count:")
    print(report["Category"].value_counts())

    print("\nDetailed Activity:")
    print("-" * 70)
    print(report.to_string(index=False))

    # Save activity log
    report.to_csv(
        "/content/file_activity_report.csv",
        index=False
    )

    print("\nReport saved as:")
    print("/content/file_activity_report.csv")

else:

    print("\nNo file-system changes detected.")

print("\nMonitoring completed.")
print("=" * 70)

             FILE ACTIVITY RECORDER

Directory : /content/monitor_folder
Duration  : 30 seconds

Monitoring started...

                 ACTIVITY SUMMARY

No file-system changes detected.

Monitoring completed.


**Result**

The Python-based file activity recorder successfully monitored the selected directory and detected created, modified, and deleted files during the specified monitoring period. Each event was recorded with its timestamp, event category, and file path, and an activity summary was generated at the end. The recorded events were also exported to a CSV file, providing a structured log that can support basic digital forensic investigation and file-system activity analysis.